# TP : Introduction au Deep Learning

## Vue d'ensemble

Ce TP couvre les 4 grandes architectures du deep learning moderne :

| # | Architecture | Framework | Dataset | Tâche |
|---|---|---|---|---|
| 1 | **Neural Network (MLP)** | PyTorch | MagicTelescope | Classification binaire |
| 2 | **CNN** | PyTorch | Fashion-MNIST | Classification d'images |
| 3 | **RNN (LSTM)** | PyTorch | Séries temporelles | Prédiction de séquence |
| 4 | **Transformer (BERT)** | HuggingFace | SST-2 | Analyse de sentiment |

## Pourquoi le Deep Learning ?

Les algorithmes classiques (Random Forest, XGBoost…) sont très efficaces sur des **données tabulaires**.  
Le deep learning excelle quand les données sont :
- **Non structurées** : images, texte, audio, vidéo
- **Très volumineuses** : la performance continue de croître avec plus de données
- **Hiérarchiques** : des patterns complexes sont composés de patterns plus simples

## Concept fondamental : le neurone artificiel

Un neurone reçoit des entrées $x_1, \ldots, x_n$, les combine linéairement, puis applique une **fonction d'activation** :

$$\text{sortie} = f\left(\sum_{i=1}^n w_i x_i + b\right)$$

Les **poids** $w_i$ et le **biais** $b$ sont appris par **rétropropagation du gradient** (backpropagation).

---
## Prérequis
Voir `pip install -r requirements.txt` a la racine de `cours_ml/todo/`.

> **Version exercice** : les cellules marquees `# TODO` sont a completer (definitions de modeles PyTorch et boucles d'entrainement). Le reste (imports, chargement des donnees, affichages, partie HuggingFace) est deja fourni.
> Installe les dependances une seule fois avec `pip install -r requirements.txt` depuis `cours_ml/todo/` (voir le README de ce dossier). Le corrige complet se trouve dans `cours_ml/04_deep_learning/tp_deep_learning.ipynb`.


---
## 0. Imports & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torch.nn.functional as F

# Sklearn (données + prétraitement)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# HuggingFace
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# GPU si disponible, sinon CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

---
# PARTIE 1 : Réseau de Neurones (MLP)
## Dataset : MagicTelescope | Framework : PyTorch

## 1.1 Qu'est-ce qu'un MLP ?

Un **Multi-Layer Perceptron (MLP)** ou réseau de neurones dense est composé de **couches de neurones** empilées :

```
Entrée (10 features)
    ↓
[Couche cachée 1 : 64 neurones] + ReLU
    ↓
[Couche cachée 2 : 32 neurones] + ReLU
    ↓
[Couche de sortie : 1 neurone] + Sigmoid
    ↓
Probabilité (0 = hadron, 1 = gamma)
```

**Fonctions d'activation :**
- **ReLU** : $f(x) = \max(0, x)$ : la plus utilisée dans les couches cachées (simple, évite le gradient qui disparaît)
- **Sigmoid** : $f(x) = \frac{1}{1+e^{-x}}$ : sortie entre 0 et 1, idéale pour la classification binaire

**Backpropagation :** à chaque itération, on calcule l'erreur (loss), puis on propage le gradient en sens inverse pour ajuster tous les poids.

**Epoch vs Batch :**
- **Epoch** : une passe complète sur toutes les données d'entraînement
- **Batch** : sous-ensemble de données traité en une fois (compromis vitesse/stabilité)
- **Mini-batch SGD** : on met à jour les poids après chaque batch (standard actuel)


## 1.2 Préparation des données

In [ ]:
# MagicTelescope : 19020 evenements captes par un telescope Cherenkov, 10 mesures -> rayon gamma ou hadron
# Chargement depuis OpenML (dataset public, aucune authentification requise)
from sklearn.datasets import fetch_openml as _fetch_openml_mlp

magic = _fetch_openml_mlp(name='MagicTelescope', version='active', as_frame=True, parser='auto')
X = magic.frame.drop(columns=['class:', 'ID'], errors='ignore').astype(float).values
y = (magic.frame['class:'].astype(str) == 'g').astype(int).values  # 1 = gamma (signal), 0 = hadron (bruit)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Conversion en tenseurs PyTorch
# PyTorch travaille avec des tenseurs (équivalent des arrays numpy, mais sur GPU)
X_train_t = torch.FloatTensor(X_train_sc)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)  # shape (N,) → (N, 1)
X_test_t  = torch.FloatTensor(X_test_sc)
y_test_t  = torch.FloatTensor(y_test).unsqueeze(1)

# DataLoader : gère les batches automatiquement
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(f"Train : {X_train_t.shape} | Test : {X_test_t.shape}")
print(f"Nombre de batches par epoch : {len(train_loader)}")


## 1.3 Définition du modèle

In [ ]:
class MLP(nn.Module):
    """
    Réseau de neurones dense (Multi-Layer Perceptron).
    En PyTorch, tout modèle hérite de nn.Module.
    """
    def __init__(self, input_dim, hidden1=64, hidden2=32):
        super().__init__()
        # TODO : definir les couches du reseau
        # - fc1 : nn.Linear(input_dim, hidden1)
        # - fc2 : nn.Linear(hidden1, hidden2)
        # - fc3 : nn.Linear(hidden2, 1)
        # - dropout : nn.Dropout(p=0.3) pour regulariser (evite l'overfitting)
        # - bn1, bn2 : nn.BatchNorm1d(hidden1) et nn.BatchNorm1d(hidden2) pour stabiliser l'entrainement
        ...

    def forward(self, x):
        """
        forward() définit le passage avant (calcul des prédictions).
        PyTorch calcule le backward (gradients) automatiquement.
        """
        # TODO : enchainer fc1 -> bn1 -> relu -> dropout -> fc2 -> bn2 -> relu -> dropout -> fc3 -> sigmoid
        ...

model_mlp = MLP(input_dim=10).to(DEVICE)
print(model_mlp)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_mlp.parameters()):,}")


## 1.4 Entraînement

La **boucle d'entraînement** PyTorch suit toujours le même schéma :
1. `optimizer.zero_grad()` : remet les gradients à zéro
2. `output = model(batch)` : passage avant
3. `loss = criterion(output, target)` : calcul de l'erreur
4. `loss.backward()` : rétropropagation (calcul des gradients)
5. `optimizer.step()` : mise à jour des poids

In [ ]:
# BCELoss : Binary Cross-Entropy : loss standard pour classification binaire
# Elle mesure l'écart entre la probabilité prédite et la vraie étiquette
criterion = nn.BCELoss()

# Adam : optimiseur adaptatif (learning rate ajusté par paramètre)
# Plus robuste que SGD pur, paramètre lr à régler
optimizer = optim.Adam(model_mlp.parameters(), lr=1e-3, weight_decay=1e-4)

# ReduceLROnPlateau : réduit le lr si la validation ne s'améliore plus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

N_EPOCHS = 100
train_losses, val_losses, val_accs = [], [], []

for epoch in range(N_EPOCHS):
    # ── Mode entraînement (active Dropout et BatchNorm)
    model_mlp.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        # TODO : boucle d'entrainement standard PyTorch
        # 1) optimizer.zero_grad()  2) preds = model_mlp(X_batch)  3) loss = criterion(preds, y_batch)
        # 4) loss.backward()        5) optimizer.step()
        ...
        batch_losses.append(loss.item())
    train_losses.append(np.mean(batch_losses))

    # ── Mode évaluation (désactive Dropout et BatchNorm)
    model_mlp.eval()
    with torch.no_grad():  # pas de calcul de gradient → plus rapide
        val_preds = model_mlp(X_test_t.to(DEVICE))
        val_loss  = criterion(val_preds, y_test_t.to(DEVICE)).item()
        val_labels = (val_preds.cpu().numpy() > 0.5).astype(int).flatten()
        val_acc = accuracy_score(y_test, val_labels)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step(val_loss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | Train loss: {train_losses[-1]:.4f} | Val loss: {val_loss:.4f} | Val acc: {val_acc:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train loss', color='steelblue')
axes[0].plot(val_losses,   label='Val loss',   color='crimson')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('MLP : Courbes de loss')
axes[0].legend()
# Si train loss << val loss → overfitting (le modèle mémorise au lieu d'apprendre)

axes[1].plot(val_accs, color='seagreen')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('MLP : Accuracy validation')
axes[1].axhline(max(val_accs), color='red', linestyle='--', label=f'Max = {max(val_accs):.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

MAGIC_CLASSES = ['hadron', 'gamma']
print(f"\nMLP : Accuracy finale : {val_accs[-1]:.4f}")
print(classification_report(y_test, val_labels, target_names=MAGIC_CLASSES))

## 1.5 Impact du learning rate : comparaison

Le **learning rate** (lr) contrôle la taille des pas lors de la descente de gradient.  
- Trop grand → le modèle diverge ou oscille  
- Trop petit → convergence très lente  
- Optimal → convergence rapide et stable

In [ ]:
lr_values = [1e-4, 1e-3, 1e-2, 1e-1]
lr_histories = {}

for lr in lr_values:
    m = MLP(input_dim=10).to(DEVICE)
    opt = optim.Adam(m.parameters(), lr=lr)
    losses = []
    for epoch in range(50):
        m.train()
        ep_loss = []
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = criterion(m(Xb), yb)
            loss.backward()
            opt.step()
            ep_loss.append(loss.item())
        losses.append(np.mean(ep_loss))
    lr_histories[lr] = losses

plt.figure(figsize=(9, 4))
for lr, losses in lr_histories.items():
    plt.plot(losses, label=f'lr={lr}')
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('Impact du learning rate sur la convergence')
plt.legend()
plt.tight_layout()
plt.show()

---
## 1.6 Explicabilité, choix du modèle final, stockage et inférence

### Explicabilité avec SHAP

Le MLP est un modèle boîte noire : contrairement à un arbre ou une régression, ses poids ne se lisent pas directement. Les **valeurs de Shapley** permettent malgré tout d'expliquer chacune de ses prédictions, feature par feature, sans changer le modèle.

In [ ]:
import shap

feature_names_mlp = magic.frame.drop(columns=['class:', 'ID'], errors='ignore').columns.tolist()

# TODO : envelopper le modele dans une fonction predict qui prend et renvoie du numpy
# Indice : model_mlp.eval() puis, dans un bloc torch.no_grad(), renvoyer
# model_mlp(torch.FloatTensor(x).to(DEVICE)).cpu().numpy().flatten()
def predict_fn(x):
    ...

background = shap.sample(X_train_sc, 50)
# TODO : creer un shap.Explainer(predict_fn, background, feature_names=feature_names_mlp)
# puis l'appliquer sur X_test_sc[:100] pour obtenir shap_values
explainer = ...
shap_values = ...

shap.summary_plot(shap_values)

**Lecture du graphique (summary plot) :** chaque point est une observation du jeu de test. La position horizontale indique l'impact sur la probabilité prédite (vers la droite : pousse vers "gamma", vers la gauche : pousse vers "hadron"), et la couleur la valeur de la feature (rouge = valeur élevée, bleu = valeur faible). Avec seulement 10 features, `shap.Explainer` choisit généralement `ExactExplainer` (énumération exacte, rapide).

In [ ]:
# TODO : tracer le waterfall plot de la premiere observation du jeu de test
# Indice : shap_values[0] (l'objet renvoye par shap.Explainer est indexable par observation)
...

print(f"Vraie classe : {MAGIC_CLASSES[y_test[0]]} | Prédite : {MAGIC_CLASSES[val_labels[0]]}")

### Choisir le modèle final

Contrairement au TP d'apprentissage supervisé (section 02, plusieurs algorithmes comparés), on n'a ici qu'un seul modèle candidat : le MLP. Le choix du modèle final revient à **valider** les trois critères avant la mise en production, plutôt qu'à arbitrer entre plusieurs candidats.

In [ ]:
# TODO : imprimer un resume de validation avant mise en production :
# 1. Erreur (1 - accuracy finale, val_accs[-1])  2. Performance (accuracy finale)  3. Explicabilite (via SHAP, ci-dessus)
# puis juger si la performance est suffisante (seuil indicatif : 0.8 -- discriminer gamma/hadron est plus difficile
# que la classification de tumeurs du TP supervise)
...

### Stocker le modèle final

In [ ]:
import joblib
import os

os.makedirs('modeles', exist_ok=True)
# TODO : sauvegarder les poids du MLP avec torch.save (state_dict) et le scaler avec joblib.dump
...
...

print("Modèle sauvegardé : modeles/best_mlp.pt (poids uniquement)")
print("Scaler sauvegardé : modeles/scaler_mlp.pkl")

En PyTorch, on sauvegarde généralement les **poids** (`state_dict`), pas l'objet Python complet : c'est plus portable et plus sûr. Recharger le modèle nécessite de réinstancier la même classe `MLP` avant d'y charger les poids.

### Inférence simple, sans API

In [ ]:
# Nouvelles données à prédire (ici, un échantillon du jeu de test, pour l'exemple :
# en production ces lignes viendraient d'une nouvelle source, pas du jeu de test)
nouvelles_donnees = pd.DataFrame(X_test[:10], columns=feature_names_mlp)

# TODO : reinstancier un MLP(input_dim=10), charger les poids sauvegardes (torch.load + load_state_dict), mode eval()
model_charge = ...
model_charge.load_state_dict(...)
model_charge.eval()

# TODO : charger le scaler sauvegarde et standardiser nouvelles_donnees
scaler_charge = ...
X_new_sc = ...

with torch.no_grad():
    probabilites = model_charge(torch.FloatTensor(X_new_sc).to(DEVICE)).cpu().numpy().flatten()

nouvelles_donnees['probabilite_gamma'] = probabilites.round(4)
nouvelles_donnees['prediction'] = [MAGIC_CLASSES[int(p > 0.5)] for p in probabilites]
nouvelles_donnees.to_csv('predictions_mlp.csv', index=False)

print(f"Prédictions sauvegardées : predictions_mlp.csv ({len(nouvelles_donnees)} lignes)")
nouvelles_donnees[['prediction', 'probabilite_gamma']]

---
# PARTIE 2 : CNN (Convolutional Neural Network)
## Dataset : Fashion-MNIST | Framework : PyTorch

## 2.1 Qu'est-ce qu'un CNN ?

Un **CNN** est un réseau conçu pour traiter des données avec une **structure spatiale** (images, sons, signaux).

**Idée clé :** au lieu de connecter chaque pixel à chaque neurone (trop de paramètres), on utilise des **filtres** qui glissent sur l'image et détectent des patterns locaux (bords, coins, textures…).

```
Image 28×28 (1 canal)
    ↓
[Conv2d : 32 filtres 3×3] + ReLU  → détecte bords et formes simples
    ↓
[MaxPooling 2×2]                  → réduit la taille (14×14), garde l'essentiel
    ↓
[Conv2d : 64 filtres 3×3] + ReLU  → détecte formes plus complexes
    ↓
[MaxPooling 2×2]                  → 7×7
    ↓
[Flatten]                         → vecteur 1D
    ↓
[Fully Connected] → 10 classes
```

**Avantages vs MLP sur images :**
- **Partage de poids** : un même filtre détecte un pattern partout dans l'image → bien moins de paramètres
- **Invariance** : reconnaît un vêtement même décalé ou légèrement transformé
- **Hiérarchie** : couches successives = features de plus en plus abstraites


## 2.2 Chargement MNIST

In [ ]:
# Fashion-MNIST : 70 000 images d'articles vestimentaires 28×28 pixels, 10 classes
from sklearn.datasets import fetch_openml

FASHION_CLASSES = ['T-shirt/top', 'Pantalon', 'Pull', 'Robe', 'Manteau',
                    'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']

print("Téléchargement Fashion-MNIST (peut prendre ~1 min la première fois)...")
fmnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False, parser='auto')
X_mnist = fmnist.data.astype(np.float32) / 255.0  # normalisation : pixels entre 0 et 1
y_mnist = fmnist.target.astype(int)

# On prend un sous-ensemble pour la rapidité
N_TRAIN, N_TEST = 10000, 2000
idx = np.random.permutation(len(X_mnist))
X_tr = X_mnist[idx[:N_TRAIN]].reshape(-1, 1, 28, 28)   # shape : (N, canal, hauteur, largeur)
y_tr = y_mnist[idx[:N_TRAIN]]
X_te = X_mnist[idx[N_TRAIN:N_TRAIN+N_TEST]].reshape(-1, 1, 28, 28)
y_te = y_mnist[idx[N_TRAIN:N_TRAIN+N_TEST]]

print(f"Train : {X_tr.shape} | Test : {X_te.shape}")
print(f"Classes : {FASHION_CLASSES}")

# Visualisation de quelques exemples
fig, axes = plt.subplots(2, 8, figsize=(14, 4.5))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(X_tr[i, 0], cmap='gray')
    ax.set_title(FASHION_CLASSES[y_tr[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Exemples Fashion-MNIST')
plt.tight_layout()
plt.show()


In [ ]:
# Conversion en tenseurs PyTorch
X_tr_t = torch.FloatTensor(X_tr)
y_tr_t = torch.LongTensor(y_tr)   # LongTensor pour les classes entières (CrossEntropy)
X_te_t = torch.FloatTensor(X_te)
y_te_t = torch.LongTensor(y_te)

mnist_train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
print(f"Batches par epoch : {len(mnist_train_loader)}")

## 2.3 Définition du CNN

In [ ]:
class CNN(nn.Module):
    """
    CNN pour classification d'images Fashion-MNIST (28×28 pixels, 10 classes).
    """
    def __init__(self):
        super().__init__()
        # TODO : definir les couches du reseau
        # Bloc convolutif 1 : conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1), bn1 = nn.BatchNorm2d(32)
        # pool = nn.MaxPool2d(2)  (28x28 -> 14x14 puis 14x14 -> 7x7)
        # Bloc convolutif 2 : conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1), bn2 = nn.BatchNorm2d(64)
        # Classificateur : fc1 = nn.Linear(64*7*7, 128), fc2 = nn.Linear(128, 10), dropout = nn.Dropout(0.5)
        ...

    def forward(self, x):
        # TODO : Bloc 1 (conv1 -> bn1 -> relu -> pool), Bloc 2 (conv2 -> bn2 -> relu -> pool),
        # aplatir avec x.view(x.size(0), -1), puis fc1 -> relu -> dropout -> fc2 (logits bruts)
        ...

model_cnn = CNN().to(DEVICE)
print(model_cnn)

# Vérification des shapes
with torch.no_grad():
    dummy = torch.randn(4, 1, 28, 28).to(DEVICE)
    out = model_cnn(dummy)
    print(f"\nInput shape  : {dummy.shape}")
    print(f"Output shape : {out.shape}  (batch=4, 10 classes)")
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_cnn.parameters()):,}")


## 2.4 Entraînement CNN

In [ ]:
# CrossEntropyLoss : loss standard pour classification multi-classes
# Combine LogSoftmax + NLLLoss → à utiliser avec des logits bruts (pas de softmax en sortie)
criterion_cnn = nn.CrossEntropyLoss()
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=1e-3)

N_EPOCHS_CNN = 10
train_losses_cnn, val_accs_cnn = [], []

for epoch in range(N_EPOCHS_CNN):
    model_cnn.train()
    ep_losses = []
    for X_batch, y_batch in mnist_train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        # TODO : meme boucle d'entrainement standard PyTorch que pour le MLP
        # zero_grad -> forward (model_cnn) -> loss (criterion_cnn) -> backward -> step
        ...
        ep_losses.append(loss.item())
    train_losses_cnn.append(np.mean(ep_losses))

    model_cnn.eval()
    with torch.no_grad():
        logits = model_cnn(X_te_t.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu().numpy()
        acc    = accuracy_score(y_te, preds)
    val_accs_cnn.append(acc)
    print(f"Epoch {epoch+1:2d}/{N_EPOCHS_CNN} | Loss: {train_losses_cnn[-1]:.4f} | Acc: {acc:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses_cnn, 'b-o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].set_title('CNN : Loss d\'entraînement')

axes[1].plot(val_accs_cnn, 'g-o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN : Accuracy validation')
axes[1].axhline(max(val_accs_cnn), color='red', linestyle='--', label=f'Max = {max(val_accs_cnn):.4f}')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2.5 Visualisation des filtres et des erreurs

In [ ]:
# Visualisation des 32 filtres de la première couche convolutive
# Chaque filtre détecte un type de pattern (bord horizontal, vertical, diagonal...)
filters = model_cnn.conv1.weight.data.cpu().numpy()  # shape : (32, 1, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, f in zip(axes.flat, filters):
    ax.imshow(f[0], cmap='RdBu_r', vmin=-1, vmax=1)
    ax.axis('off')
plt.suptitle('Filtres de la 1ère couche convolutive (3×3 pixels chacun)')
plt.tight_layout()
plt.show()

In [ ]:
# Exemples mal classifiés
model_cnn.eval()
with torch.no_grad():
    logits_all = model_cnn(X_te_t.to(DEVICE))
    preds_all  = logits_all.argmax(1).cpu().numpy()

errors = np.where(preds_all != y_te)[0]
print(f"Erreurs : {len(errors)} / {len(y_te)} ({len(errors)/len(y_te):.1%})")

fig, axes = plt.subplots(2, 8, figsize=(14, 5))
for ax, idx in zip(axes.flat, errors[:16]):
    ax.imshow(X_te[idx, 0], cmap='gray')
    ax.set_title(f'Vrai:{FASHION_CLASSES[y_te[idx]]}\nPrédit:{FASHION_CLASSES[preds_all[idx]]}', fontsize=7)
    ax.axis('off')
plt.suptitle('Exemples mal classifiés')
plt.tight_layout()
plt.show()


---
# PARTIE 3 : RNN / LSTM
## Dataset : Série temporelle | Framework : PyTorch

## 3.1 Qu'est-ce qu'un RNN ?

Un **Réseau de Neurones Récurrent (RNN)** traite des **séquences** en maintenant une **mémoire cachée** qui se propage de pas en pas :

```
x₁ → [RNN] → h₁ → ŷ₁
x₂ → [RNN] → h₂ → ŷ₂    (h₁ transmis à h₂)
x₃ → [RNN] → h₃ → ŷ₃    (h₂ transmis à h₃)
```

**Problème du RNN simple :** le gradient s'évanouit ou explose sur de longues séquences.  
→ Solution : **LSTM** (Long Short-Term Memory)

## Le LSTM

Le LSTM ajoute une **cellule mémoire** $c_t$ contrôlée par 3 **portes** (gates) :

| Porte | Rôle |
|---|---|
| **Forget gate** $f_t$ | Qu'est-ce qu'on oublie de la mémoire passée ? |
| **Input gate** $i_t$ | Qu'est-ce qu'on mémorise de la nouvelle entrée ? |
| **Output gate** $o_t$ | Qu'est-ce qu'on utilise de la mémoire pour la sortie ? |

Ces portes permettent au LSTM de **capturer des dépendances à long terme** dans la séquence.

**Applications :** prédiction de séries temporelles, traduction automatique, génération de texte, reconnaissance vocale.

## 3.2 Génération des données temporelles

In [ ]:
# Tâche : prédire le prochain pas d'une série temporelle composée de sin + bruit
# Le modèle voit une fenêtre de SEQ_LEN pas et doit prédire le suivant

SEQ_LEN   = 30   # longueur de la séquence d'entrée
N_SAMPLES = 2000

t = np.linspace(0, 8 * np.pi, N_SAMPLES + SEQ_LEN)
# Série multi-fréquence + bruit → plus réaliste qu'un simple sinus
series = np.sin(t) + 0.5 * np.sin(3 * t) + 0.1 * np.random.randn(len(t))

# Normalisation
series = (series - series.mean()) / series.std()

# Construction des fenêtres glissantes
X_seq = np.array([series[i:i+SEQ_LEN]         for i in range(N_SAMPLES)])  # (N, SEQ_LEN)
y_seq = np.array([series[i+SEQ_LEN]           for i in range(N_SAMPLES)])  # (N,)

# Split
split = int(0.8 * N_SAMPLES)
X_seq_tr, X_seq_te = X_seq[:split], X_seq[split:]
y_seq_tr, y_seq_te = y_seq[:split], y_seq[split:]

# Tenseurs : shape (N, SEQ_LEN, 1) : LSTM attend (batch, sequence, features)
X_seq_tr_t = torch.FloatTensor(X_seq_tr).unsqueeze(2)
y_seq_tr_t = torch.FloatTensor(y_seq_tr).unsqueeze(1)
X_seq_te_t = torch.FloatTensor(X_seq_te).unsqueeze(2)
y_seq_te_t = torch.FloatTensor(y_seq_te).unsqueeze(1)

seq_loader = DataLoader(TensorDataset(X_seq_tr_t, y_seq_tr_t), batch_size=64, shuffle=True)

# Visualisation
plt.figure(figsize=(12, 3))
plt.plot(t[:200], series[:200], color='steelblue')
plt.title('Série temporelle (200 premiers pas)')
plt.xlabel('Temps')
plt.ylabel('Valeur')
plt.tight_layout()
plt.show()

print(f"Train : {X_seq_tr_t.shape} | Test : {X_seq_te_t.shape}")

## 3.3 Définition du modèle LSTM

In [ ]:
class LSTMPredictor(nn.Module):
    """
    LSTM pour la prédiction de série temporelle.
    Entrée  : séquence de longueur SEQ_LEN
    Sortie  : prédiction du prochain pas
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        # TODO : definir un nn.LSTM(input_size=input_size, hidden_size=hidden_size,
        # num_layers=num_layers, batch_first=True, dropout=dropout)
        # puis une couche de sortie fc = nn.Linear(hidden_size, 1)
        self.lstm = ...
        self.fc = ...

    def forward(self, x):
        # TODO : passer x dans le LSTM, ne garder que le dernier pas de la sequence,
        # puis appliquer fc dessus
        # Indice : out, (h_n, c_n) = self.lstm(x) ; last_hidden = out[:, -1, :]
        ...

model_lstm = LSTMPredictor().to(DEVICE)
print(model_lstm)
print(f"\nNombre de paramètres : {sum(p.numel() for p in model_lstm.parameters()):,}")


## 3.4 Entraînement LSTM

In [ ]:
# MSELoss : Mean Squared Error : loss standard pour la régression
criterion_lstm = nn.MSELoss()
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=1e-3)

N_EPOCHS_LSTM = 30
train_losses_lstm, val_losses_lstm = [], []

for epoch in range(N_EPOCHS_LSTM):
    model_lstm.train()
    ep_loss = []
    for Xb, yb in seq_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        # TODO : meme boucle d'entrainement (zero_grad -> forward -> loss -> backward -> step)
        # Ajoute torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1.0) avant optimizer_lstm.step()
        # pour eviter l'explosion du gradient (probleme classique des RNN)
        ...
        ep_loss.append(loss.item())
    train_losses_lstm.append(np.mean(ep_loss))

    model_lstm.eval()
    with torch.no_grad():
        val_loss = criterion_lstm(model_lstm(X_seq_te_t.to(DEVICE)), y_seq_te_t.to(DEVICE)).item()
    val_losses_lstm.append(val_loss)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{N_EPOCHS_LSTM} | Train MSE: {train_losses_lstm[-1]:.5f} | Val MSE: {val_loss:.5f}")


In [ ]:
# Prédictions vs réalité
model_lstm.eval()
with torch.no_grad():
    y_pred_lstm = model_lstm(X_seq_te_t.to(DEVICE)).cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Courbes de loss
axes[0].plot(train_losses_lstm, label='Train MSE')
axes[0].plot(val_losses_lstm,   label='Val MSE')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('LSTM : Courbes de loss')
axes[0].legend()

# Prédictions sur les 150 premiers pas de test
n_plot = 150
axes[1].plot(y_seq_te[:n_plot], label='Réel', color='steelblue')
axes[1].plot(y_pred_lstm[:n_plot], label='Prédit', color='crimson', linestyle='--')
axes[1].set_xlabel('Pas de temps')
axes[1].set_ylabel('Valeur')
axes[1].set_title('LSTM : Prédictions vs Réalité')
axes[1].legend()

plt.tight_layout()
plt.show()

from sklearn.metrics import mean_squared_error, r2_score
print(f"LSTM : MSE test : {mean_squared_error(y_seq_te, y_pred_lstm):.5f}")
print(f"LSTM : R²  test : {r2_score(y_seq_te, y_pred_lstm):.4f}")

## 3.5 Comparaison LSTM vs modèle naïf

Un **modèle naïf** prédit simplement la dernière valeur connue (`ŷₜ = xₜ₋₁`).  
C'est la baseline à battre pour tout modèle de série temporelle.

In [ ]:
# Modèle naïf : prédit la dernière valeur de la séquence
y_naive = X_seq_te[:, -1]   # dernier pas de chaque fenêtre

print(f"{'Modèle':<20} {'MSE':>10} {'R²':>10}")
print('-' * 42)
print(f"{'Naïf (last value)':<20} {mean_squared_error(y_seq_te, y_naive):>10.5f} {r2_score(y_seq_te, y_naive):>10.4f}")
print(f"{'LSTM':<20} {mean_squared_error(y_seq_te, y_pred_lstm):>10.5f} {r2_score(y_seq_te, y_pred_lstm):>10.4f}")

---
# PARTIE 4 : Transformer (BERT)
## Dataset : SST-2 (sentiments) | Framework : HuggingFace Transformers

## 4.1 Qu'est-ce qu'un Transformer ?

Les **Transformers** (2017, "Attention is All You Need") ont révolutionné le NLP et au-delà.  
Contrairement aux RNNs, ils traitent la séquence **en parallèle** grâce au mécanisme d'**attention**.

**Mécanisme d'attention (Self-Attention) :**  
Chaque mot regarde tous les autres mots de la séquence et calcule une **pondération** pour chacun.  
→ "Le chat mange la souris" : pour comprendre "mange", le modèle "fait attention" à "chat" et "souris".

```
Texte tokenisé : [CLS] "The" "film" "was" "great" [SEP]
       ↓
[Embedding : chaque token → vecteur de 768 dimensions]
       ↓
[12 couches Transformer avec Multi-Head Self-Attention]
       ↓
[Token [CLS] = représentation globale de la phrase]
       ↓
[Tête de classification] → positif / négatif
```

## BERT vs GPT

| | **BERT** | **GPT** |
|---|---|---|
| Type | Encodeur | Décodeur |
| Entraînement | Masked Language Model (MLM) | Next Token Prediction |
| Bidirectionnel | Oui (voit le contexte gauche ET droit) | Non (gauche seulement) |
| Usage principal | Classification, NER, QA | Génération de texte |

## Transfer Learning en NLP

BERT est **pré-entraîné** sur des milliards de mots (Wikipedia, livres…) pendant des semaines.  
On peut ensuite **fine-tuner** ce modèle sur une tâche spécifique avec peu de données en quelques minutes.  
C'est le principe du **transfer learning** : réutiliser une connaissance générale pour une tâche spécifique.

## 4.2 Pipeline HuggingFace : Analyse de sentiment

In [ ]:
# HuggingFace pipeline : utilisation du modèle en une ligne
# Le modèle distilbert-base-uncased-finetuned-sst-2-english est un BERT allégé
# déjà fine-tuné sur l'analyse de sentiment (SST-2)
# Premier appel : télécharge le modèle (~250 MB)
print("Chargement du modèle BERT (distilBERT fine-tuné sur SST-2)...")
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)
print("Modèle chargé !")

In [ ]:
# Test sur des phrases exemples
test_sentences = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "The film was boring and the plot made no sense whatsoever.",
    "I have mixed feelings about this one. Some parts were great, others not so much.",
    "An incredible masterpiece, one of the best films I have ever seen.",
    "Terrible acting, terrible script, I want my two hours back.",
    "Not bad, but not great either. Decent way to spend an evening.",
]

results = sentiment_pipeline(test_sentences)

print(f"{'Phrase':<60} {'Label':<12} {'Score':>8}")
print('-' * 82)
for sentence, result in zip(test_sentences, results):
    short = sentence[:57] + '...' if len(sentence) > 60 else sentence
    emoji = '😊' if result['label'] == 'POSITIVE' else '😞'
    print(f"{short:<60} {result['label']:<12} {result['score']:>8.4f} {emoji}")

## 4.3 Évaluation sur un dataset de référence (SST-2)

In [ ]:
# Dataset SST-2 : critiques de films annotées POSITIVE / NEGATIVE
# On crée un échantillon représentatif pour l'évaluation
sst2_samples = [
    ("a stirring , funny and finally transporting re-imagining of beauty and the beast", 1),
    ("apparently reassembled from the cutting-room floor of any given daytime soap", 0),
    ("they\u2019re moldy and old and not much fun", 0),
    ("the picture never takes off", 0),
    ("allows us to hope that nolan is poised to embark a major career as a commercial", 1),
    ("the script is short on logic", 0),
    ("the acting is uniformly good", 1),
    ("it\u2019s a lovely , lively film", 1),
    ("a dumb movie", 0),
    ("the movie is hilarious , touching , warm", 1),
    ("an extremely unpleasant film", 0),
    ("a masterful film", 1),
    ("the movie is a mess", 0),
    ("a wonderful little animation", 1),
    ("this is a fascinating film", 1),
    ("this is a boring film", 0),
    ("heartfelt and funny", 1),
    ("it barely even makes sense", 0),
    ("so beautifully crafted", 1),
    ("deeply unpleasant", 0),
]

texts_sst2  = [s[0] for s in sst2_samples]
labels_sst2 = [s[1] for s in sst2_samples]

preds_raw = sentiment_pipeline(texts_sst2)
preds_sst2 = [1 if r['label'] == 'POSITIVE' else 0 for r in preds_raw]
scores_sst2 = [r['score'] for r in preds_raw]

acc_bert = accuracy_score(labels_sst2, preds_sst2)
print(f"Accuracy BERT sur SST-2 (échantillon) : {acc_bert:.4f}")
print()
print(classification_report(labels_sst2, preds_sst2, target_names=['NEGATIVE', 'POSITIVE']))

In [ ]:
# Visualisation des scores de confiance
df_bert = pd.DataFrame({
    'Texte': [t[:40] + '...' for t in texts_sst2],
    'Vrai label': ['POS' if l == 1 else 'NEG' for l in labels_sst2],
    'Prédit': ['POS' if p == 1 else 'NEG' for p in preds_sst2],
    'Score': scores_sst2,
    'Correct': [l == p for l, p in zip(labels_sst2, preds_sst2)]
})

colors = ['seagreen' if c else 'crimson' for c in df_bert['Correct']]
plt.figure(figsize=(10, 6))
plt.barh(df_bert['Texte'], df_bert['Score'], color=colors, edgecolor='k', linewidth=0.5)
plt.axvline(0.5, color='black', linestyle='--')
plt.xlabel('Score de confiance')
plt.title('BERT : Scores de sentiment (vert = correct, rouge = erreur)')
plt.tight_layout()
plt.show()

## 4.4 Comprendre la tokenisation BERT

In [ ]:
# Le tokenizer transforme le texte en IDs compréhensibles par le modèle
# BERT utilise WordPiece : les mots rares sont découpés en sous-unités
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

exemples = [
    "This movie was fantastic!",
    "The cinematography was breathtaking.",
]

for phrase in exemples:
    tokens = tokenizer.tokenize(phrase)
    ids    = tokenizer.encode(phrase)
    print(f"Phrase  : {phrase}")
    print(f"Tokens  : {tokens}")
    print(f"IDs     : {ids}")
    print(f"  → [CLS]={ids[0]}, tokens={ids[1:-1]}, [SEP]={ids[-1]}")
    print()

## 4.5 GPT : Génération de texte

Là où BERT **comprend** le texte (encodeur bidirectionnel), GPT le **génère** (décodeur autorégressif).

**Principe autorégressif :** GPT prédit le mot suivant à partir de tous les mots précédents, puis recommence en intégrant le mot qu'il vient de produire :

```
"Le chat" → "mange"
"Le chat mange" → "la"
"Le chat mange la" → "souris"
```

**Paramètres de génération clés :**
- `max_length` : longueur maximale de la séquence générée
- `temperature` : contrôle l'aléatoire : basse (0.2) = prévisible/répétitif, haute (1.0+) = créatif/risqué
- `top_k` / `top_p` : restreignent le vocabulaire candidat à chaque étape (nucleus sampling)
- `num_return_sequences` : nombre de variantes générées

GPT-2 est la version open et légère de la famille GPT : parfaite pour comprendre le mécanisme sans ressources massives.

In [ ]:
# Chargement de GPT-2 (~500 MB au premier appel)
from transformers import pipeline, set_seed

print("Chargement de GPT-2...")
generator = pipeline('text-generation', model='gpt2')
set_seed(RANDOM_STATE)  # reproductibilité de la génération
print("Modèle chargé !")

In [ ]:
# Génération à partir d'une amorce (prompt)
prompt = "Artificial intelligence will"

outputs = generator(
    prompt,
    max_length=50,
    num_return_sequences=3,   # 3 continuations différentes
    temperature=0.8,
    top_k=50,
    pad_token_id=generator.tokenizer.eos_token_id,
)

print(f"Prompt : '{prompt}'\n")
for i, out in enumerate(outputs, 1):
    print(f"--- Génération {i} ---")
    print(out['generated_text'])
    print()

## 4.6 Effet de la température

La **température** est le levier le plus parlant pour comprendre la génération.  
On génère le même prompt avec différentes valeurs pour observer l'effet sur la créativité et la cohérence.

In [ ]:
prompt = "The future of technology is"
temperatures = [0.2, 0.7, 1.2]

for temp in temperatures:
    set_seed(RANDOM_STATE)
    out = generator(
        prompt,
        max_length=40,
        num_return_sequences=1,
        temperature=temp,
        top_k=50,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    print(f"--- Température = {temp} ---")
    print(out[0]['generated_text'])
    print()

# Température basse  → texte sûr mais répétitif
# Température haute  → texte varié mais parfois incohérent

---
# PARTIE 5 : Comparaison & Bilan

## 5.1 Résumé des performances

In [ ]:
# Récupération des scores finaux
mlp_acc  = val_accs[-1]
cnn_acc  = val_accs_cnn[-1]
lstm_r2  = r2_score(y_seq_te, y_pred_lstm)
bert_acc = acc_bert

summary = pd.DataFrame([
    {'Architecture': 'MLP',  'Tâche': 'Classification binaire', 'Dataset': 'MagicTelescope', 'Métrique': 'Accuracy', 'Score': round(mlp_acc, 4)},
    {'Architecture': 'CNN',  'Tâche': 'Classification images',  'Dataset': 'Fashion-MNIST',         'Métrique': 'Accuracy', 'Score': round(cnn_acc, 4)},
    {'Architecture': 'LSTM', 'Tâche': 'Régression temporelle',  'Dataset': 'Sinusoïde',     'Métrique': 'R²',       'Score': round(lstm_r2, 4)},
    {'Architecture': 'BERT', 'Tâche': 'Analyse de sentiment',   'Dataset': 'SST-2',         'Métrique': 'Accuracy', 'Score': round(bert_acc, 4)},
    {'Architecture': 'GPT-2','Tâche': 'Génération de texte',     'Dataset': ': (pré-entraîné)','Métrique': ':',        'Score': np.nan},
])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Nombre de paramètres
params = {
    'MLP\n(MagicTelescope)':  sum(p.numel() for p in model_mlp.parameters()),
    'CNN\n(Fashion-MNIST)':          sum(p.numel() for p in model_cnn.parameters()),
    'LSTM\n(Série temp.)':   sum(p.numel() for p in model_lstm.parameters()),
    'BERT\n(Sentiment)':     66_000_000,  # distilBERT ~66M paramètres
}
axes[0].bar(params.keys(), params.values(), color=['steelblue', 'darkorange', 'seagreen', 'purple'], edgecolor='k')
axes[0].set_ylabel('Nombre de paramètres')
axes[0].set_title('Complexité des modèles')
axes[0].set_yscale('log')
for ax_bar, (name, val) in zip(axes[0].patches, params.items()):
    axes[0].text(ax_bar.get_x() + ax_bar.get_width()/2, ax_bar.get_height() * 1.2,
                 f'{val:,}', ha='center', va='bottom', fontsize=8)

# Score
colors_arch = ['steelblue', 'darkorange', 'seagreen', 'purple']
bars = axes[1].bar(summary['Architecture'], summary['Score'], color=colors_arch, edgecolor='k')
for bar, score in zip(bars, summary['Score']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{score:.4f}', ha='center', va='bottom', fontsize=10)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score (Accuracy ou R²)')
axes[1].set_title('Performance par architecture')

plt.tight_layout()
plt.show()

---
## 5.2 Quand utiliser quoi ?

| Architecture | Données | Points forts | À éviter |
|---|---|---|---|
| **MLP** | Tabulaires, vecteurs fixes | Simple, rapide, interprétable | Images, séquences, texte |
| **CNN** | Images, signaux 1D/2D | Invariance spatiale, peu de paramètres | Données sans structure spatiale |
| **RNN/LSTM** | Séquences temporelles, texte | Mémoire, ordre naturel | Très longues séquences (→ Transformer) |
| **Transformer** | Texte, images (ViT), multimodal | State-of-the-art universel | Peu de données, ressources limitées |

## 5.3 Concepts clés à retenir

**Architecture :**
- Plus de couches = plus expressif, mais plus difficile à entraîner
- Batch Normalization et Dropout sont des régulariseurs essentiels

**Entraînement :**
- Toujours surveiller les courbes train vs validation (diagnostic overfitting/underfitting)
- Learning rate est l'hyperparamètre le plus important
- Adam est un bon point de départ, SGD + momentum peut faire mieux avec le bon tuning

**Transfer Learning :**
- Pour texte → partir d'un modèle pré-entraîné (BERT, GPT) plutôt que from scratch
- Pour images → partir de modèles pré-entraînés sur ImageNet (ResNet, EfficientNet)

## 5.4 Pistes d'approfondissement

- **CNN avancé** : ResNet, EfficientNet avec transfer learning sur une image réelle
- **Attention** : implémenter le mécanisme d'attention from scratch
- **Fine-tuning BERT** : entraîner la tête de classification sur un dataset custom
- **GPT avancé** : tester GPT-2 large, ou contrôler la génération avec top_p (nucleus sampling)
- **Vision Transformer (ViT)** : Transformers appliqués aux images

---
## Session à rendre

Cette section est à compléter et à rendre à l'issue du TP. Réponds à chaque question dans la
cellule *Réponse* juste en dessous, à partir des résultats que **tu as toi-même obtenus** en
réalisant ce notebook (il n'y a pas de réponse générique valable pour tout le monde : les valeurs
numériques, choix d'hyperparamètres et graphiques dépendent de ton exécution).

**Q1.** Quelle accuracy finale obtiens-tu sur le MLP (MagicTelescope) ? Observes-tu de l'overfitting sur les courbes train vs validation loss ?

*Réponse :*

_(à compléter)_

**Q2.** D'après ton graphique de comparaison des learning rates, quel learning rate converge le mieux pour ce MLP ? Que se passe-t-il visuellement pour lr=1e-1 ?

*Réponse :*

_(à compléter)_

**Q3.** Quelle accuracy finale obtiens-tu sur le CNN (Fashion-MNIST) ?

*Réponse :*

_(à compléter)_

**Q4.** En observant les exemples mal classifiés par le CNN, quelles catégories de vêtements sont le plus souvent confondues entre elles, et pourquoi selon toi ?

*Réponse :*

_(à compléter)_

**Q5.** Quel R² obtiens-tu avec le LSTM sur la prédiction de série temporelle ? Bat-il le modèle naïf (dernière valeur connue) ? De combien ?

*Réponse :*

_(à compléter)_

**Q6.** Quelle accuracy obtiens-tu avec BERT (distilBERT) sur l'échantillon SST-2, sans aucun entraînement de ta part ? Qu'est-ce que cela illustre sur le transfer learning ?

*Réponse :*

_(à compléter)_

**Q7.** En comparant les tailles (nombre de paramètres) et les performances des 4 architectures dans ton tableau récapitulatif, quel modèle offre le meilleur rapport performance/complexité pour sa tâche ?

*Réponse :*

_(à compléter)_

**Q8.** Sur le summary plot SHAP du MLP, quelles variables (mesures du télescope) ont le plus d'impact sur la distinction gamma/hadron ?

*Réponse :*

_(à compléter)_

**Q9.** La performance du MLP a-t-elle été jugée suffisante pour la mise en production (section 1.6) ? Sur quel seuil t'es-tu basé ?

*Réponse :*

_(à compléter)_

**Q10.** Sur les 10 nouvelles prédictions sauvegardées dans `predictions_mlp.csv`, combien sont classées comme "gamma" ?

*Réponse :*

_(à compléter)_